# SEPO Stage 1 — SFT Warm Start (Colab)
**Model**: `google/gemma-3-4b-it`  
**Method**: QLoRA (4-bit base + LoRA rank 8)  
**Data**: `sepo_sft_data/` — 6400 IPD strategy demonstrations  
**Output**: PEFT adapter uploaded to `kartiinx/gemma-3-4b-sepo-sft-hf`

Runtime: **T4 GPU** — `Runtime > Change runtime type > T4`

In [ ]:
# ── Cell 1: Check GPU ─────────────────────────────────────────────────────────
!nvidia-smi

In [ ]:
# ── Cell 2: Install dependencies ──────────────────────────────────────────────
!pip install -q transformers accelerate peft bitsandbytes trl huggingface_hub datasets

In [ ]:
# ── Cell 3: HuggingFace login ─────────────────────────────────────────────────
from huggingface_hub import login
login()  # paste HF token with read+write access

In [ ]:
# ── Cell 4: Clone repo (grpo-stage2 branch) ───────────────────────────────────
GITHUB_TOKEN = "YOUR_GITHUB_TOKEN_HERE"  # github.com/settings/tokens → classic → repo scope
!git clone -b grpo-stage2 https://{GITHUB_TOKEN}@github.com/kirankumarmanku/sepo.git
%cd sepo

In [ ]:
# ── Cell 5: Verify data ───────────────────────────────────────────────────────
import json

with open("sepo_sft_data/train.jsonl") as f:
    samples = [json.loads(l) for l in f]

print(f"Train examples: {len(samples)}")
print("Sample keys:", list(samples[0].keys()))
print("\nSample messages:")
for m in samples[0]["messages"]:
    print(f"  [{m['role']}]: {m['content'][:80]}...")

In [ ]:
# ── Cell 6: Load model + tokenizer (4-bit QLoRA) ──────────────────────────────
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import get_peft_model, LoraConfig, TaskType

MODEL_ID = "google/gemma-3-4b-it"

bnb_4bit = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_4bit,
    device_map="auto",
)

# LoRA config — rank 8 matches local MLX training
lora_cfg = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.0,
    bias="none",
)
model = get_peft_model(base_model, lora_cfg)
model.print_trainable_parameters()
print(f"VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# ── Cell 8: SFT Training ──────────────────────────────────────────────────────
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = "/content/sft_gemma3_ipd"

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    bf16=False,
    fp16=True,
    gradient_checkpointing=True,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_steps=200,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    processing_class=tokenizer,
    max_seq_length=512,             # passed to SFTTrainer directly
)

print("Starting SFT training...")
trainer.train()

In [ ]:
# ── Cell 8: SFT Training ──────────────────────────────────────────────────────
from trl import SFTTrainer, SFTConfig
from pathlib import Path

OUTPUT_DIR = "/content/sft_gemma3_ipd"

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,     # effective batch = 4
    learning_rate=1e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    bf16=False,                         # T4 doesn't support bf16
    fp16=True,
    gradient_checkpointing=True,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_steps=200,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    max_seq_length=512,
    dataset_text_field=None,            # use messages format directly
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    processing_class=tokenizer,
)

print("Starting SFT training...")
trainer.train()

In [ ]:
# ── Cell 9: Check val loss curve — pick best checkpoint ───────────────────────
# Best checkpoint is auto-loaded (load_best_model_at_end=True)
# Look at the logged eval_loss values above.
# We want the checkpoint where val loss ~0.01 (not 0.000 — that's overfit)
print("Training complete. Best model loaded.")
print(f"VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# ── Cell 10: Save adapter + upload to HuggingFace ─────────────────────────────
# Creates a NEW HF repo: kartiinx/gemma-3-4b-sepo-sft-hf
# This is a clean PEFT adapter (HF-compatible, unlike the MLX-fused version)

ADAPTER_REPO = "kartiinx/gemma-3-4b-sepo-sft-hf"

# Save adapter locally
model.save_pretrained(f"{OUTPUT_DIR}/final_adapter")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final_adapter")

# Push to HF Hub (creates private repo automatically)
model.push_to_hub(ADAPTER_REPO, private=True)
tokenizer.push_to_hub(ADAPTER_REPO, private=True)

print(f"Adapter uploaded to: {ADAPTER_REPO}")
print("Use this repo as --model in grpo_sepo_colab.ipynb")

In [ ]:
# ── Cell 11: Quick sanity check — generate a sample action ────────────────────
from peft import PeftModel

model.eval()
test_messages = [
    {"role": "system", "content": "You are playing the Iterated Prisoner's Dilemma game.\n\nRules:\n- Each round you choose one of two actions: <SILENT> or <TESTIFY>\n- If both players choose <SILENT>: you each get 3 points\n- If you choose <TESTIFY> and opponent chooses <SILENT>: you get 5, opponent gets 0\n- If you choose <SILENT> and opponent chooses <TESTIFY>: you get 0, opponent gets 5\n- If both choose <TESTIFY>: you each get 1 point\n\nYour goal is to maximise your total score over all rounds.\nRespond with ONLY your action: <SILENT> or <TESTIFY>. Nothing else."},
    {"role": "user", "content": "Round 1 of 8.\nThis is the first round. No history yet.\n\nWhat is your action?"}
]

input_ids = tokenizer.apply_chat_template(
    test_messages, return_tensors="pt", add_generation_prompt=True
).to("cuda")

with torch.no_grad():
    out = model.generate(input_ids, max_new_tokens=8, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id)

response = tokenizer.decode(out[0, input_ids.shape[1]:], skip_special_tokens=True)
print(f"Model response: '{response}'")
print("Expected: <SILENT> (cooperate — SEPO-optimal for round 1)")